### server.py

In [ ]:
async def handle_define(request: web.Request) -> web.Response:
    """
    Обрабатывает запрос на определение алгоритмов хеширования.

    request: Объект запроса aiohttp.
    return: JSON со списком возможных алгоритмов.
    """
    hash_value = request.match_info.get('hash', '')
    algorithms = identify_hashes(hash_value)
    return web.json_response(algorithms)

Объявляем корутину (потому что используем 2 ручки, а они требуют асинхронного выполнения). Результат должен быть выведен как web.Response. На входе ожидаем объект класса web.Request

Словарь с параметрами из динамических сегментов маршрута (например, из /hash/{hash} параметр hash). безопасное извлечение значения по ключу 'hash'. Если ключ отсутствует, возвращается пустая строка '' (вместо исключения KeyError). 

Вызывает функцию identify_hashes, передавая ей значение хеша. 

web.json_response() — вспомогательная функция aiohttp, которая:
- Принимает Python-объект (здесь algorithms — список строк).
- Сериализует его в JSON.
- Создаёт объект web.Response с заголовком Content-Type: application/json.
- Возвращает этот готовый HTTP-ответ клиенту.



In [ ]:
async def handle_solve(request: web.Request) -> web.Response:
    """
    Обрабатывает запрос на 'раскрытие' хеша по конкретному алгоритму.

    request: Объект запроса aiohttp.
    return: JSON с результатом вычисления.
    """
    hash_value = request.query.get('hash', '')
    algorithm = request.query.get('algorithm', '')

    if not hash_value or not algorithm:
        return web.json_response(
            {'error': 'Missing hash or algorithm parameters'},
            status=400
        )

    result = await long_solve_hash(hash_value, algorithm)
    return web.json_response({'hash': hash_value, 'result': result})

Объявляем вторую корутину, на входе ожидаем объект класса web.Request, на выходе аннотируем объект класса web.Response

- request.query — словарь с параметрами URL-запроса (из строки запроса после ?). Например, для URL /solve?hash=abc123&algorithm=md5
- .get('hash', '') — безопасное получение значения параметра hash. Если параметр отсутствует — возвращается пустая строка ''

- Аналогично получает параметр algorithm (алгоритм хеширования: 'md5', 'sha1', 'sha256' и т.д.)
- При отсутствии параметра — пустая строка

Проверка валидности входных данных. Условие срабатывает, если хотя бы один из параметров:
- Пустая строка ('')
- None
- Любое "ложное" (falsy) значение

Возврат ошибки валидации при отсутствии обязательных параметров
- web.json_response() — создаёт JSON-ответ с телом {"error": "Missing hash or algorithm parameters"}
- status=400 — HTTP статус 400 Bad Request (неверный запрос)

- await — асинхронное ожидание результата (не блокирует выполнение других задач)
- long_solve_hash() восстанавливает секретное слово
- Результат сохраняется в переменную result

Формирует успешный JSON-ответ с двумя полями:
- 'hash' — исходный хеш (передаётся обратно для идентификации запроса)
- 'result' — результат расшифровки секретного слова
- HTTP статус по умолчанию — 200 OK

In [ ]:
def main():
    """Настройка и запуск приложения aiohttp."""
    app = web.Application()
    app.add_routes([
        web.get('/define/{hash}', handle_define),
        web.get('/solve', handle_solve),
    ])
    web.run_app(app, host='127.0.0.1', port=8080)

- web.Application() — создание экземпляра приложения aiohttp
- app — объект приложения, который:
    - Хранит маршруты (routes)
    - Управляет middleware
    - Содержит настройки и состояние приложения
    - Является центральным компонентом веб-сервера

app.add_routes() — метод для добавления маршрутов (роутов) в приложение. Принимает список (или кортеж) маршрутов

web.get() — создаёт маршрут для HTTP метода GET
- '/define/{hash}' — URL-шаблон:
- Статическая часть: /define/
- Динамический параметр: {hash} — будет доступен как request.match_info['hash']
- handle_define — функция-обработчик

web.get() — маршрут для GET-запросов
- '/solve' — статический URL (без параметров в пути)
- handle_solve — обработчик для этого маршрута

web.run_app() — функция для запуска веб-сервера (блокирует выполнение до остановки)
- app — созданное ранее приложение
- host='127.0.0.1' — сетевой интерфейс для прослушивания:
- 127.0.0.1 (localhost) — доступно только с этого же компьютера
- port=8080 — TCP-порт
- Функция запускает:
- TCP-сервер на указанном хосте и порте
- Обработку входящих запросов
- Обычно работает бесконечно до нажатия Ctrl+C

### client.py

In [ ]:
async def fetch_algorithms(session: aiohttp.ClientSession,
                           hash_value: str) -> list:
    """
    Получает список алгоритмов для хеша с сервера.

    session: Сессия aiohttp.
    hash_value: Строка хеша.
    return: Список строк (алгоритмов).
    """
    url = f'http://127.0.0.1:8080/define/{hash_value}'
    async with session.get(url) as response:
        return await response.json()

Объявляем асинхронную корутину, на входе ожидаем сессию и значение хэша, на выходе - список алгоритмов, который сервер определил для этого хэша

Форматированная строка (f-string) — вставляет значение hash_value в URL
- Формирует полный URL для запроса:
- http://127.0.0.1:8080 — тот же сервер (localhost, порт 8080)
- /define/ — эндпоинт из server.py
- {hash_value} — подставляется хеш

async with — асинхронный контекстный менеджер:
- Автоматически управляет жизненным циклом соединения
- Не блокирует другие задачи во время ожидания
- Гарантирует закрытие соединения при выходе из блока (даже при ошибках)
- session.get(url) — выполняет асинхронный HTTP GET-запрос
- as response — сохраняет объект ответа в переменную response

response.json() — асинхронный метод, читающий тело ответа и парсящий его из JSON в Python-объект
- await — ожидает завершения чтения и парсинга (может занять время, если ответ большой)
- return — немедленно возвращает результат

In [ ]:
async def solve_hash(session: aiohttp.ClientSession,
                     hash_value: str, algorithm: str) -> dict:
    """
    Отправляет запрос на решение хеша по конкретному алгоритму.

    session: Сессия aiohttp.
    hash_value: Строка хеша.
    algorithm: Название алгоритма.
    return: Словарь с результатом.
    """
    url = 'http://127.0.0.1:8080/solve'
    params = {'hash': hash_value, 'algorithm': algorithm}
    async with session.get(url, params=params) as response:
        return await response.json()

Объявляем корутину, на входе ожидаем сессию, значение хэша и уже определенный алгоритм, на выходе возвращаем словарь, в котором ключ - значение хэша, значение - алгоритм

Статический URL (без параметров в пути)
- Указывает на эндпоинт /solve сервера
- Параметры будут переданы через строку запроса (query string)

Словарь параметров для GET-запроса
- Ключи: 'hash' и 'algorithm' — ожидаемые сервером для передачи в long_solve_hash()
- Значения: переданные в функцию параметры

async with — асинхронный контекстный менеджер для управления соединением
- session.get(url, params=params) — GET-запрос с параметрами:
- Параметры автоматически кодируются и добавляются к URL
- as response — сохраняет объект ответа в переменную

response.json() — асинхронное чтение и парсинг JSON из тела ответа
- await — ожидание получения и обработки данных

In [ ]:
async def main():
    """Основная логика работы клиента."""
    target_hash = 'c4ca4238a0b923820dcc509a6f75849b'

    async with aiohttp.ClientSession() as session:
        print(f"Идентификация хеша: {target_hash}")
        algorithms = await fetch_algorithms(session, target_hash)

        if not algorithms:
            print("Алгоритмы не найдены.")
            return

        print(f"Найденные алгоритмы: {', '.join(algorithms)}")
        print("Запуск вычислений для каждого алгоритма...")

        # Создаем список задач для конкурентного выполнения
        tasks = [
            solve_hash(session, target_hash, alg)
            for alg in algorithms
        ]

        # Ожидаем завершения всех задач
        results = await asyncio.gather(*tasks)

        for res in results:
            print(f"Результат: {res.get('result')}")

Объявляем асинхронную корутину, которая является основной функцией

Указываем целевой хэш для расшифровки (в данном случае это типичный md5)

async with — асинхронный контекстный менеджер
- aiohttp.ClientSession() — создание HTTP-клиентской сессии
- as session — сохраняет сессию в переменную
- Автоматически закрывает сессию при выходе из блока
- Важно: Одна сессия на все запросы (эффективное переиспользование соединений)

Вывод информационного сообщения в консоль
- Показывает, какой хеш анализируется

await fetch_algorithms(...) — асинхронный вызов функции определения алгоритмов
- Передаёт сессию и целевой хеш
- Возвращает список алгоритмов
- Ожидает ответа от сервера

Если алгоритмы не найдены, делаем об этом уведомление и заканчиваем выполнение функции

', '.join(algorithms) — объединяет список алгоритмов в строку через запятую
- Выводит найденные алгоритмы для информации

Информирует о начале этапа расшифровки

List comprehension (списковое включение)
- Для каждого алгоритма из списка algorithms:
- Создаёт корутину solve_hash(session, target_hash, alg)
- Важно: Корутины ещё не запущены, только созданы
- tasks — список корутин (объектов ожидающих выполнения)

asyncio.gather(*tasks) — запускает все задачи конкурентно (параллельно)
- *tasks — распаковывает список в отдельные аргументы
- Запускает корутины одновременно, а не последовательно
- Ждёт завершения всех задач
- await — ожидание окончания всех задач
- results — список результатов в том же порядке, что и задачи

res.get('result') — безопасное извлечение значения по ключу 'result'
- Если ключ отсутствует — вернёт None (без исключения)
- Выводит расшифрованное значение (или None, если не найдено)

In [ ]:
if __name__ == '__main__':
    asyncio.run(main())

Создаем event loop, запускаем корутину main(), управляем ее жизненным циклом, после завершения закрываем